In [0]:
%pip install sentence-transformers faiss-cpu langchain langchain-community

In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

faiss_index_path = "/Volumes/workspace/365pdf/365pdfupload/faiss_intro_tableau_index"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.load_local(
    faiss_index_path,
    embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS vector index loaded successfully")

In [0]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created successfully")

In [0]:
def ask_pdf(question):
    retrieved_docs = retriever.invoke(question)

    print("Question:")
    print(question)
    print("\n" + "=" * 80)
    print("Retrieved context from PDF:")
    print("=" * 80)

    for i, doc in enumerate(retrieved_docs, start=1):
        print(f"\nSource {i}")
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print("Source file:", doc.metadata.get("source"))
        print("-" * 80)
        print(doc.page_content[:1200])
        print("-" * 80)

    combined_context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    answer = f"""
Based on the uploaded PDF, here is the most relevant information I found:

{combined_context[:2500]}

Sources used:
{[doc.metadata for doc in retrieved_docs]}
"""

    return answer

In [0]:
answer = ask_pdf("What is Tableau used for?")
print(answer)

In [0]:
answer = ask_pdf("What are dimensions and measures in Tableau?")
print(answer)

In [0]:
answer = ask_pdf("How do you create a visualization in Tableau?")
print(answer)

In [0]:
def ask_pdf_with_guardrail(question, min_results=1):
    retrieved_docs = retriever.invoke(question)

    if len(retrieved_docs) < min_results:
        return "I could not find relevant information in the uploaded PDF."

    source_lines = []
    context_parts = []

    for i, doc in enumerate(retrieved_docs, start=1):
        chunk_id = doc.metadata.get("chunk_id")
        source = doc.metadata.get("source")

        source_lines.append(f"Source {i}: {source}, chunk_id={chunk_id}")
        context_parts.append(doc.page_content)

    combined_context = "\n\n".join(context_parts)

    response = f"""
Answer based only on the uploaded PDF:

{combined_context[:2500]}

Sources:
{chr(10).join(source_lines)}
"""

    return response

In [0]:
print(ask_pdf_with_guardrail("What is Tableau used for?"))

In [0]:
print(ask_pdf_with_guardrail("What is the capital of Japan?"))